In [1]:
import pandas as pd, numpy as np

DIR = "/kaggle/input/competitions/playground-series-s6e10/"
train = pd.read_csv(DIR + "train.csv")
test = pd.read_csv(DIR + "test.csv")
sub = pd.read_csv(DIR + "sample_submission.csv")

print(train.shape, test.shape, sub.shape)
print(train["satisfaction"].value_counts(normalize=True))
print(train.isna().sum()[train.isna().sum() > 0])
print(train.dtypes.value_counts())

(699635, 23) (299844, 22) (299844, 2)
satisfaction
False    0.556427
True     0.443573
Name: proportion, dtype: float64
Arrival Delay in Minutes    292
dtype: int64
int64      17
object      4
float64     1
bool        1
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import StratifiedKFold

y = train["satisfaction"].astype(int)
X = train.drop(columns=["id", "satisfaction"])
X_test = test.drop(columns=["id"])

cat_cols = X.select_dtypes("object").columns.tolist()
print(cat_cols)

# Make categories consistent across train and test
for c in cat_cols:
    cats = pd.Categorical(pd.concat([X[c], X_test[c]])).categories
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X.shape, X_test.shape, y.mean())

['Gender', 'Customer Type', 'Type of Travel', 'Class']
(699635, 21) (299844, 21) 0.44357272006117476


In [3]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=50,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    n_estimators=3000,
    verbose=-1,
    n_jobs=-1,
)

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    oof[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold}: AUC {roc_auc_score(y.iloc[va], oof[va]):.5f} | trees {model.best_iteration_}")

print("OOF AUC:", roc_auc_score(y, oof))

np.save("oof_lgb.npy", oof)
np.save("test_lgb.npy", test_pred)

submission = pd.DataFrame({"id": test["id"], "satisfaction": test_pred})
submission.to_csv("submission.csv", index=False)

Fold 0: AUC 0.95910 | trees 248
Fold 1: AUC 0.95775 | trees 264
Fold 2: AUC 0.95983 | trees 308
Fold 3: AUC 0.95874 | trees 360
Fold 4: AUC 0.95935 | trees 230
OOF AUC: 0.9589390312565133


In [4]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

/kaggle/input []
/kaggle/input/competitions []
/kaggle/input/competitions/playground-series-s6e10 ['sample_submission.csv', 'train.csv', 'test.csv']
/kaggle/input/datasets []
/kaggle/input/datasets/teejmahal20 []
/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction ['train.csv', 'test.csv']


In [5]:
ODIR = "/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction/"
orig = pd.concat(
    [pd.read_csv(ODIR + "train.csv"), pd.read_csv(ODIR + "test.csv")],
    ignore_index=True,
)
print(orig.shape)
print(orig.columns.tolist())

# Drop index/id columns, convert label to 0/1
orig = orig.drop(columns=[c for c in orig.columns if c.lower().startswith("unnamed") or c == "id"])
print(orig["satisfaction"].unique())
orig["satisfaction"] = (orig["satisfaction"] == "satisfied").astype(int)

# Check it lines up with our features
print("Missing in orig:", set(X.columns) - set(orig.columns))
print("Extra in orig:", set(orig.columns) - set(X.columns) - {"satisfaction"})

# Same categorical handling as before
y_orig = orig["satisfaction"]
X_orig = orig[X.columns].copy()
for c in cat_cols:
    X_orig[c] = pd.Categorical(X_orig[c], categories=X[c].cat.categories)

print(X_orig.shape, y_orig.mean())
print(X_orig[cat_cols].isna().sum())

(129880, 25)
['Unnamed: 0', 'id', 'Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']
['neutral or dissatisfied' 'satisfied']
Missing in orig: set()
Extra in orig: {'Inflight service'}
(129880, 21) 0.43446258084385586
Gender            0
Customer Type     0
Type of Travel    0
Class             0
dtype: int64


In [6]:
oof2 = np.zeros(len(X))
test_pred2 = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    X_tr = pd.concat([X.iloc[tr], X_orig], ignore_index=True)
    y_tr = pd.concat([y.iloc[tr], y_orig], ignore_index=True)

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    oof2[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_pred2 += model.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold}: AUC {roc_auc_score(y.iloc[va], oof2[va]):.5f} | trees {model.best_iteration_}")

print("OOF AUC with original data:", roc_auc_score(y, oof2))

np.save("oof_lgb_orig.npy", oof2)
np.save("test_lgb_orig.npy", test_pred2)

Fold 0: AUC 0.95890 | trees 492
Fold 1: AUC 0.95741 | trees 253
Fold 2: AUC 0.95931 | trees 340
Fold 3: AUC 0.95868 | trees 376
Fold 4: AUC 0.95905 | trees 342
OOF AUC with original data: 0.9586723315308978


In [7]:
rating_cols = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service", "Leg room service",
    "Baggage handling", "Checkin service", "Cleanliness",
]

def add_features(df):
    df = df.copy()
    r = df[rating_cols]
    df["rating_mean"] = r.mean(axis=1)
    df["rating_sum"] = r.sum(axis=1)
    df["rating_std"] = r.std(axis=1)
    df["rating_min"] = r.min(axis=1)
    df["rating_max"] = r.max(axis=1)
    df["n_fives"] = (r == 5).sum(axis=1)
    df["n_ones"] = (r <= 1).sum(axis=1)
    df["n_low"] = (r <= 2).sum(axis=1)
    df["n_zero"] = (r == 0).sum(axis=1)
    df["delay_total"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]
    df["delay_diff"] = df["Arrival Delay in Minutes"] - df["Departure Delay in Minutes"]
    df["delay_per_km"] = df["delay_total"] / (df["Flight Distance"] + 1)
    combo = (df["Class"].astype(str) + "|" + df["Type of Travel"].astype(str)
             + "|" + df["Customer Type"].astype(str))
    df["combo"] = pd.Categorical(combo)
    return df

full = add_features(pd.concat([X, X_test], ignore_index=True))
X_fe = full.iloc[:len(X)].reset_index(drop=True)
X_test_fe = full.iloc[len(X):].reset_index(drop=True)
print(X_fe.shape, X_test_fe.shape)

oof3 = np.zeros(len(X_fe))
test_pred3 = np.zeros(len(X_test_fe))

for fold, (tr, va) in enumerate(skf.split(X_fe, y)):
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_fe.iloc[tr], y.iloc[tr],
        eval_set=[(X_fe.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    oof3[va] = model.predict_proba(X_fe.iloc[va])[:, 1]
    test_pred3 += model.predict_proba(X_test_fe)[:, 1] / skf.n_splits
    print(f"Fold {fold}: AUC {roc_auc_score(y.iloc[va], oof3[va]):.5f} | trees {model.best_iteration_}")

print("OOF AUC with features:", roc_auc_score(y, oof3))

np.save("oof_lgb_fe.npy", oof3)
np.save("test_lgb_fe.npy", test_pred3)

(699635, 34) (299844, 34)
Fold 0: AUC 0.95900 | trees 250
Fold 1: AUC 0.95765 | trees 273
Fold 2: AUC 0.95937 | trees 240
Fold 3: AUC 0.95858 | trees 213
Fold 4: AUC 0.95912 | trees 226
OOF AUC with features: 0.9587335441258383


In [8]:
from scipy.stats import rankdata

params3 = dict(
    objective="binary", metric="auc",
    learning_rate=0.05, num_leaves=31, min_child_samples=200,
    feature_fraction=0.6, bagging_fraction=0.7, bagging_freq=1,
    lambda_l1=1.0, lambda_l2=10.0, cat_smooth=50,
    n_estimators=2000, verbose=-1, n_jobs=-1,
)

oof5 = np.zeros(len(X))
test_pred5 = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    model = lgb.LGBMClassifier(**params3)
    model.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    oof5[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_pred5 += model.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold}: AUC {roc_auc_score(y.iloc[va], oof5[va]):.5f} | trees {model.best_iteration_}")

print("OOF AUC regularized:", roc_auc_score(y, oof5))

# Blend with the models already trained (rank average)
r = lambda a: rankdata(a) / len(a)
oofs = [oof, oof2, oof3, oof5]
tests = [test_pred, test_pred2, test_pred3, test_pred5]

blend_oof = np.mean([r(o) for o in oofs], axis=0)
blend_test = np.mean([r(t) for t in tests], axis=0)
print("OOF AUC blend of 4:", roc_auc_score(y, blend_oof))

# Pick the best of: blend vs. regularized alone
best_test = blend_test if roc_auc_score(y, blend_oof) >= roc_auc_score(y, oof5) else test_pred5

submission = pd.DataFrame({"id": test["id"], "satisfaction": best_test})
submission.to_csv("submission.csv", index=False)
print(submission.head(), submission.shape)

Fold 0: AUC 0.95910 | trees 1099
Fold 1: AUC 0.95765 | trees 996
Fold 2: AUC 0.95952 | trees 776
Fold 3: AUC 0.95893 | trees 1251
Fold 4: AUC 0.95919 | trees 643
OOF AUC regularized: 0.9588668116677053
OOF AUC blend of 4: 0.9591966140383196
       id  satisfaction
0  699635      0.523591
1  699636      0.608188
2  699637      0.638585
3  699638      0.428956
4  699639      0.355783 (299844, 2)


In [9]:
import xgboost as xgb

xparams = dict(
    tree_method="hist",
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="auc",
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=10.0,
    reg_alpha=1.0,
    n_estimators=2000,
    early_stopping_rounds=100,
    n_jobs=-1,
)

oof6 = np.zeros(len(X))
test_pred6 = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    m = xgb.XGBClassifier(**xparams)
    m.fit(X.iloc[tr], y.iloc[tr],
          eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
    oof6[va] = m.predict_proba(X.iloc[va])[:, 1]
    test_pred6 += m.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold}: AUC {roc_auc_score(y.iloc[va], oof6[va]):.5f} | trees {m.best_iteration}")

print("OOF AUC xgboost:", roc_auc_score(y, oof6))

# Re-blend with all 5 models
oofs5 = oofs + [oof6]
tests5 = tests + [test_pred6]
blend_oof5 = np.mean([r(o) for o in oofs5], axis=0)
blend_test5 = np.mean([r(t) for t in tests5], axis=0)

new_auc = roc_auc_score(y, blend_oof5)
print("OOF AUC blend of 5:", new_auc, "| previous blend: 0.95920")

if new_auc > 0.959197:
    pd.DataFrame({"id": test["id"], "satisfaction": blend_test5}).to_csv("submission.csv", index=False)
    print("submission.csv updated")

Fold 0: AUC 0.95903 | trees 613
Fold 1: AUC 0.95778 | trees 925
Fold 2: AUC 0.95970 | trees 821
Fold 3: AUC 0.95889 | trees 744
Fold 4: AUC 0.95913 | trees 670
OOF AUC xgboost: 0.9588949342615711
OOF AUC blend of 5: 0.9592367106394767 | previous blend: 0.95920
submission.csv updated
